# Datos

In [6]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_rows', None)

# Cargar el dataset
print("Buscando ruta del dataset...")
ruta_csv = 'global_concatenado/global_concatenado.CSV'
#si existe el archivo en la ruta especificada
try:    
    with open(ruta_csv, 'r') as file:
        print("Archivo encontrado.")
except FileNotFoundError:
    print(f"Error: El archivo '{ruta_csv}' no se encontró.")
    
print("Intentando cargar el dataset...")
try:
    df = pd.read_csv(ruta_csv)
    print("Dataset cargado exitosamente.")
except FileNotFoundError:
    print(f"Error: El archivo '{ruta_csv}' no se encontró.")

print(df.head())
print(df.info())

Buscando ruta del dataset...
Archivo encontrado.
Intentando cargar el dataset...
Dataset cargado exitosamente.
   Unnamed: 0.1  Unnamed: 0 source  psi_psa1  psi_psa2  psi_psa3  psi_psa4  \
0             0           0  POXC1  61.16965  59.99485  66.90590  65.55704   
1             1           1  POXC1  61.72080  64.47652  62.18492  61.79332   
2             2           2  POXC1  66.52880  64.99865  66.77536  62.57652   
3             3           3  POXC1  62.67805  59.27691  66.49254  65.68032   
4             4           4  POXC1  60.76355  63.46850  61.97461  63.25094   

   psi_tablero   flujo  totalizador  ... suma_psa  suma_compresores  psi_psa5  \
0     49.44336  415.80    2234119.0  ...      4.0               4.0       NaN   
1     48.74717  405.84    2234125.0  ...      4.0               4.0       NaN   
2     49.78419  417.15    2234132.0  ...      4.0               4.0       NaN   
3     49.43610  416.88    2234138.0  ...      4.0               4.0       NaN   
4     40.52354 

In [8]:
# Crear un DataFrame de resumen con nulos y tipos de dato
resumen = pd.DataFrame({
    'Nulos': df.isnull().sum(),
    'Tipo_de_Dato': df.dtypes
})

# Forzar a que se muestren todas las filas
print(resumen.to_string())

                       Nulos Tipo_de_Dato
Unnamed: 0.1               0        int64
Unnamed: 0                 0        int64
source                     0          str
psi_psa1              297228      float64
psi_psa2              297228      float64
psi_psa3              434287      float64
psi_psa4              434287      float64
psi_tablero           297228      float64
flujo                 280060      float64
totalizador           297228      float64
TIME                       0          str
r_psa1                  7649      float64
r_psa2                  7649      float64
r_psa3                144708      float64
r_psa4                144708      float64
r_gen1                 21204      float64
r_gen2                 21204      float64
r_bar                  21204      float64
r_sec1                 21204      float64
r_sec2                 21204      float64
r_com1                 21204      float64
r_com2                 21204      float64
r_com3                158263      


### **¿Qué significa cada columna? **

Sabiendo que la empresa se dedica a inyectar oxígeno a los peces las empresas utilizan plantas generadoras de oxígeno in situ.

Como tienes 109 columnas, la mejor forma de entenderlas es agruparlas por su **prefijo**, ya que siguen una nomenclatura estándar de telemetría industrial (sistemas SCADA o PLCs).

**1. Identificadores y Tiempo**

* `Unnamed: 0` / `Unnamed: 0.1`: Son índices antiguos o números de fila que se guardaron por error al exportar el CSV originalmente. (Te sugiero eliminarlas después).
* `source`: El origen de los datos o el identificador del pontón/centro de cultivo (ej. POXC1).
* `TIME`: La marca de tiempo (fecha y hora) exacta en la que se tomó la medición.
* `Sistema`: Probablemente el nombre o ID general del sistema operando.

**2. Sistema PSA (Generadores de Oxígeno)**
*Las plantas generan oxígeno separándolo del aire mediante un proceso llamado PSA (Pressure Swing Adsorption).*

* `psi_psa1` a `psi_psa6`: La presión (en PSI - libras por pulgada cuadrada) de cada uno de los generadores PSA (hasta 6 equipos).
* `suma_psa`: Cantidad total de módulos PSA que están encendidos o funcionando en ese momento.

**3. Flujo y Entrega de Oxígeno**

* `psi_tablero`: La presión de oxígeno en el tablero principal de distribución, justo antes de enviarlo a las jaulas de los peces.
* `flujo`: La cantidad de oxígeno que se está inyectando en ese instante (probablemente medido en litros por minuto o metros cúbicos por hora).
* `totalizador`: El volumen acumulado total de oxígeno que se ha entregado a lo largo del tiempo (como el cuentakilómetros de un auto).

**4. Variables de Estado o Funcionamiento (`r_`)**
*La "r" generalmente viene de "Run" (en marcha/corriendo) o "Relay". Indican si un equipo está encendido (1) o apagado (0).*

* `r_psa1` a `r_psa6`: Estado de marcha de los módulos PSA.
* `r_com1` a `r_com4`: Estado de marcha de los compresores de aire (el aire comprimido alimenta a los PSA).
* `suma_compresores`: Cantidad total de compresores encendidos.
* `r_gen1`, `r_gen2`: Estado de los generadores eléctricos.
* `r_sec1`, `r_sec2`: Estado de los secadores de aire (eliminan la humedad del aire antes de que entre al PSA).
* `r_bar`: Estado de la barredora o sistema de barrido.

**5. Variables de Control (`sp_`, `ox_`, `m_`)**
*Datos provenientes de los sensores instalados (probablemente en las distintas jaulas o líneas de inyección de oxígeno, numerados del s1 al s12).*

* `sp_s1` a `sp_s12`: **Setpoint** (Punto de ajuste). Es el nivel de oxígeno o presión objetivo que el operador programó en el sistema para ese sensor.
* `ox_s1` a `ox_s12`: La medición real de **concentración o nivel de oxígeno** (pureza) que está leyendo el sensor.
* `m_s1` a `m_s12`: Probablemente **Modo** de operación (ej. Automático vs Manual) o estado de la válvula/caudalímetro de ese sensor.

**6. Temperatura**

* `mb_g1_temperatura_f` / `mb_g2_temperatura_f`: La temperatura en grados Fahrenheit (°F) de los generadores o motores 1 y 2 (el prefijo "mb" suele referirse a Modbus, el protocolo de comunicación utilizado para extraer el dato).

**7. Señales del Sistema (`hb_`, `rst_`)**

* `hb_*` (ej. `hb_psa1`, `hb_com1`): **Heartbeat** (Latido). Es una señal digital (1 o 0) que el equipo envía cada segundo para decir "Estoy conectado y en línea". Si se pierde, significa que se cortó la comunicación de internet o red con el equipo.
* `rst_*` (ej. `rst_psa1`, `rst_com1`): **Reset / Restart**. Indica si se envió una señal de reinicio a ese equipo, o la cantidad de veces que se ha reiniciado por fallas.